In [2]:
import pyomo.environ as pe
from pyomo.opt import SolverFactory, SolverStatus, TerminationCondition
from pyomo.contrib.iis import *
from pyomo.core.expr.visitor import identify_mutable_parameters, identify_variables
import cloudpickle

In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Created on Mon Jul 31 14:45:01 2023

@author: geconsta
"""

from pyomo.environ import *

# Create a model
model = ConcreteModel()

# Sets
#'aircraft types and unassigned passengers'
I = ['a', 'b', 'c', 'd']
#'assigned and unassigned routes'
J = ['route-1', 'route-2', 'route-3', 'route-4', 'route-5']
#'demand states'
H = [1, 2, 3, 4, 5]

model.i = Set(initialize=I, doc='Aircraft types and unassigned passengers')
model.j = Set(initialize=J, doc='Assigned and unassigned routes')
model.h = Set(initialize=H, doc='Demand states')
model.hp = Set(initialize=H, doc='Demand states')

# Demand distribution on route j
dd_init = {
    ('route-1', 1): 200, ('route-1', 2): 220, ('route-1', 3): 250, ('route-1', 4): 270, ('route-1', 5): 300,
    ('route-2', 1): 50,  ('route-2', 2): 150, ('route-2', 3): 0,   ('route-2', 4): 0,   ('route-2', 5): 0,
    ('route-3', 1): 140, ('route-3', 2): 160, ('route-3', 3): 180, ('route-3', 4): 200, ('route-3', 5): 220,
    ('route-4', 1): 10,  ('route-4', 2): 50,  ('route-4', 3): 80,  ('route-4', 4): 100, ('route-4', 5): 340,
    ('route-5', 1): 580, ('route-5', 2): 600, ('route-5', 3): 620, ('route-5', 4): 0,   ('route-5', 5): 0}
# probability of demand state h on route j
lambda_init = {
    ('route-1', 1): 0.2, ('route-1', 2): 0.05,('route-1', 3): 0.35,('route-1', 4): 0.2, ('route-1', 5): 0.2,
    ('route-2', 1): 0.3, ('route-2', 2): 0.7, ('route-2', 3): 0,   ('route-2', 4): 0,   ('route-2', 5): 0,
    ('route-3', 1): 0.1, ('route-3', 2): 0.2, ('route-3', 3): 0.4, ('route-3', 4): 0.2, ('route-3', 5): 0.1,
    ('route-4', 1): 0.2, ('route-4', 2): 0.2, ('route-4', 3): 0.3, ('route-4', 4): 0.2, ('route-4', 5): 0.1,
    ('route-5', 1): 0.1, ('route-5', 2): 0.8, ('route-5', 3): 0.1, ('route-5', 4): 0,   ('route-5', 5): 0}
# costs per aircraft (1000s)
c_init = {
    ('a','route-1'): 18, ('a','route-2'): 21, ('a','route-3'): 18, ('a','route-4'): 16, ('a','route-5'): 10,
    ('b','route-1'): 0,  ('b','route-2'): 15, ('b','route-3'): 16, ('b','route-4'): 14, ('b','route-5'): 9,
    ('c','route-1'): 0,  ('c','route-2'): 10, ('c','route-3'): 0,  ('c','route-4'): 9,  ('c','route-5'): 6,
    ('d','route-1'): 17, ('d','route-2'): 16, ('d','route-3'): 17, ('d','route-4'): 15, ('d','route-5'): 10}
# passenger capacity of aircraft i on route j
p_init = {
    ('a','route-1'): 16, ('a','route-2'): 15, ('a','route-3'): 28, ('a','route-4'): 23, ('a','route-5'): 81,
    ('b','route-1'): 0,  ('b','route-2'): 10, ('b','route-3'): 14, ('b','route-4'): 15, ('b','route-5'): 57,
    ('c','route-1'): 0,  ('c','route-2'): 5,  ('c','route-3'): 0,  ('c','route-4'): 7,  ('c','route-5'): 29,
    ('d','route-1'): 9,  ('d','route-2'): 11, ('d','route-3'): 22, ('d','route-4'): 17, ('d','route-5'): 55}

# Parameters


model.dd = Param(model.j, model.h, initialize=dd_init, mutable = True, doc= 'demand distribution on route j')
model.lambda_ = Param(model.j, model.h, initialize=lambda_init, mutable = True, doc= 'probability of demand state h on route j')
model.c = Param(model.i, model.j, initialize=c_init, mutable = False, doc='costs per aircraft (1000s)')
model.p = Param(model.i, model.j, initialize=p_init, mutable = True, doc='passenger capacity of aircraft i on route j')
model.aa = Param(model.i, initialize={
    'a': 10, 'b': 19, 'c': 25, 'd': 15}, mutable = True, doc='aircraft availability')
#revenue lost (1000 per 100 bumped)
model.k = Param(model.j, initialize={
    'route-1': 13, 'route-2': 13, 'route-3': 7,'route-4': 7,'route-5': 1}, mutable = True, doc='revenue lost (1000 per 100 bumped')

def ed_init(model, j):
    return sum(lambda_init[j,h]*dd_init[j,h] for h in model.h)
model.ed = Param(model.j, initialize=ed_init, mutable = True, doc='expected demand')

# def gamma_init(model, j, h):
#     return sum(lambda_init[j,hp] for hp in model.h if hp>=h)
# model.gamma = Param(model.j, model.h, initialize=gamma_init, mutable = True)

def deltb_init(model, j, h):
    if dd_init[j,h] > 0:
        if h >= 2:
            return dd_init[j,h] - dd_init[j,h-1]
        else:
            return dd_init[j,h]
    else:
        return 0
model.deltb = Param(model.j, model.h, initialize=deltb_init, mutable = True, doc='incremental passenger load in demand states')

# Variables
# number of aircraft type i assigned to route j
model.x = Var(model.i, model.j, domain=NonNegativeReals, doc='number of aircraft type i assigned to route j')
# passengers actually carried
model.y = Var(model.j, model.h, domain=NonNegativeReals, doc='passengers actually carried')
# passengers bumped
model.b = Var(model.j, model.h, domain=NonNegativeReals, doc='passengers bumped')
# operating cost
model.oc = Var(domain=NonNegativeReals, doc='operating cost')
# bumping cost
model.bc = Var(domain=NonNegativeReals, doc='bumping cost')
# total expected costs
model.phi = Var()

# Objective
model.obj = Objective(expr=model.oc + model.bc, sense=minimize, doc='objective function')

# Constraints

# model.db = Constraint(model.j, rule=lambda model, j: sum(model.p[i,j]*model.x[i,j] for i in model.i) >= sum(model.y[j,h] for h in model.h))
# model.bcd1 = Constraint(rule=lambda model: model.bc == sum(model.k[j]*(model.ed[j] - sum(model.y[j,h]*model.gamma[j,h] for h in model.h)) for j in model.j))

# aircraft balance'
model.ab = Constraint(model.i, rule=lambda model, i: sum(model.x[i,j] for j in model.j) <= model.aa[i], doc='aircraft balance')
# definition of boarded passengers
model.yd = Constraint(model.j, model.h, rule=lambda model, j, h: model.y[j, h] <= sum(model.p[i, j]*model.x[i, j] for i in model.i), doc='definition of boarded passengers')
# definition of bumped passengers
model.bd = Constraint(model.j, model.h, rule=lambda model, j, h: model.b[j, h] == model.dd[j, h] - model.y[j, h], doc='definition of bumped passengers')
# operating cost definition
model.ocd = Constraint(rule=lambda model: model.oc == sum(model.c[i, j]*model.x[i, j] for i in model.i for j in model.j), doc='operating cost definition')
# bumping cost definition: version 2
model.bcd2 = Constraint(rule=lambda model: model.bc == sum(model.k[j]*model.lambda_[j, h]*model.b[j, h] for j in model.j for h in model.h), doc='bumping cost definition: version 2')


model.yup = Constraint(model.j, model.h, rule=lambda model, j,h: model.y[j,h] <= model.deltb[j,h])

In [5]:
model.ab.expr

AttributeError: 'IndexedConstraint' object has no attribute 'expr'

In [ ]:
cfg_template = {
    "models": {"local_resources": ["my_model.pkl"],},
    "models_code": {"local_resources": ["Feas/*"]}
    }
cfg_out = expand_wildcard_in_cfg(cfg_template)
print(cfg_out)

In [ ]:
solver = SolverFactory('gurobi')
results = solver.solve(model, tee=False)
model.obj.pprint()
model.obj.display()

In [ ]:
model.results

In [ ]:
model.ab["a"].uslack()

In [ ]:
model.x[('a', 'route-1')].getname()

In [ ]:
model.x[('a', 'route-1')]

In [ ]:
pe.value(model.find_component('x[a,route-1]'))

In [ ]:
for idx in model.dd:
    dd_i = model.dd[idx]
    print(pe.value(dd_i))
    print(pe.value(model.find_component(pe.name(dd_i))))

In [ ]:
for p in model.component_objects(pe.Var, active=True):
    if p.is_indexed():
        for idx in p:
            p_i = p[idx]
            print(pe.value(p_i))
            print(pe.value(model.find_component(pe.name(p_i))))
    else:
        p_i = p
        print(pe.value(p_i))
        print(pe.value(model.find_component(pe.name(p_i))))

In [ ]:
import re
def wildcard_to_regex(pattern: str) -> str:
    """
    Convert a wildcard pattern to a regex pattern.
    The intuition of this function is to escape all regex special characters, as pyomo's component names usually contain "[", "]", "(", ")", etc.
    Then only rely on '*' and '?' for wildcard matching.

    Args:
        pattern (str): The wildcard pattern to convert.

    Returns:
        str: The corresponding regex pattern.
    """
    # Escape all regex special characters
    regex_pattern = re.escape(pattern)
    # Replace escaped wildcards with regex equivalents
    regex_pattern = regex_pattern.replace(r'\*', '.*').replace(r'\?', '.')
    # Anchor the pattern to match the whole string
    return f'^{regex_pattern}$'

def fn(pattern, candidates):
    regex_pattern = wildcard_to_regex(pattern)
    compiled_pattern = re.compile(regex_pattern, re.IGNORECASE)
    print([c for c in candidates if compiled_pattern.match(c)])
    
pattern = "config[1]*.json"
candidates = ["config[1].json", "config[1]_backup.json", "config[2].json"]
fn(pattern, candidates)

# Langchain RAG

In [ ]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders.text import TextLoader
from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.parsers import LanguageParser
from langchain_text_splitters import Language, RecursiveCharacterTextSplitter
from openai import embeddings

from optichat.config.rag_cfg import *


def init_code_rag(path: str, model_name: str):
    """
Recommended way to load source code: 
https://docs.langchain.com/oss/python/integrations/document_loaders/source_code
The parser can be disabled for small files.
This approach needs path to be a folder (or a single file) and uses glob pattern to load files.
TODO: Not sure if there exists an alternative way to load files from a list of specific files.
    """
    code_loader = GenericLoader.from_filesystem(
        path,
        glob="*",
        suffixes=[".py"],
        parser=LanguageParser(language=Language.PYTHON, parser_threshold=CODE_RAG_PARSER_THRESHOLD))
    docs = code_loader.load()

    if CODE_RAG_IS_SPLITTED:
        code_splitter = RecursiveCharacterTextSplitter.from_language(
            language=Language.PYTHON, chunk_size=CODE_RAG_CHUNK_SIZE, chunk_overlap=CODE_RAG_CHUNK_OVERLAP)
        docs = code_splitter.split_documents(docs)

    embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
    vector_store = Chroma(
        collection_name=f"{model_name}_code",
        embedding_function=embeddings,
        persist_directory="./chroma_langchain_db")
    vector_store.add_documents(documents=docs)

In [ ]:
init_code_rag("Feas/pp.py", "pp")

In [ ]:
init_code_rag("Feas/thai.py", "thai")

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
vector_store = Chroma(
    collection_name=f"pp_code",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db")

retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})
result = retriever.invoke("what is sequence 4 constraint")
print(result)

In [ ]:
result = retriever.invoke("what is sequence 4 constraint")
print(result)

In [ ]:
vector_store._client.delete_collection(name="pp_code")

In [ ]:
vector_store = Chroma(
    collection_name=f"thai_code",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db")
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})
result = retriever.invoke("what is sequence 4 constraint")
print(result)

In [ ]:
vector_store = Chroma(
    collection_name=f"pp_code",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db")
col_names = [col.name for col in vector_store._client.list_collections()]
print(col_names)

In [ ]:
vector_store = Chroma(
    collection_name=f"thai_code",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db")
col_names = [col.name for col in vector_store._client.list_collections()]
print(col_names)

In [ ]:
"pp_code" in col_names

# validate prompts

In [ ]:
this is a test prompt from developer, please do the following: root agent outputs 'hey this is root agent'

this is a test prompt from developer, please do the following: root agent use expert agent tool, expert agent doesn't use any tool but outputs 'this is optichat', root agent then explain 'optichat is the expert agent'.

this is a test prompt from developer, use expert agent and code_rag to check what the bd constraint is in the code and its meaning. return the code snippet as evidence for debugging purpose. DON'T use get_model_components.

this is a test prompt from developer, use expert agent and python_repl_func. The code snippet is
model = load_model('my_model', models_dictionary)
new_models_dictionary = solve_model(model, 'xyz', models_dictionary)
I will monitor how if you can execute this concise code snippet correctly. DON'T use get_model_components. DON'T do extra things. 

this is a test prompt from developer, use expert agent and get_model_components to search for this constraint: bd[route-4,4] in version xyz. DON'T do extra things.
